# API-based cleaning (Information Systems)

Same logic as root `datacleaning.py`: unique (Author_name, Author_Address) pairs, batched API calls, cache, 3 RPM / 200 RPD.

In [2]:
# =========================
# Cell 1 — Imports & Config
# =========================

import os
import json
import time
import csv
import re
from pathlib import Path

import pandas as pd
from dotenv import load_dotenv
from openai import OpenAI

# -------- Files --------
INPUT_CSV  = Path("Information_Systems_basic_cleaned.csv")
OUTPUT_CSV = Path("Information_Systems_API_cleaned.csv")
CACHE_JSON = Path("standardization_cache.json")

# -------- Model --------
MODEL = "gpt-4o-mini"

# -------- Rate limiting --------
RPM_LIMIT = 3
DELAY_BETWEEN_CALLS = 60.0 / RPM_LIMIT

# -------- Run limit (max API calls per run) --------
MAX_CALLS_PER_RUN = 20

# -------- Batch --------
BATCH_SIZE = 10

load_dotenv()
api_key = os.getenv("OPENAI_API_KEY")
if not api_key:
    raise ValueError("OPENAI_API_KEY not found. Add it to .env or your environment variables.")
client = OpenAI(api_key=api_key)

print("Config loaded.")
print("INPUT :", INPUT_CSV.resolve())
print("OUTPUT:", OUTPUT_CSV.resolve())
print("CACHE :", CACHE_JSON.resolve())

Config loaded.
INPUT : /Users/keerthisagi/Documents/Journals/Journal_of_Information_Systems/Information_Systems_basic_cleaned.csv
OUTPUT: /Users/keerthisagi/Documents/Journals/Journal_of_Information_Systems/Information_Systems_API_cleaned.csv
CACHE : /Users/keerthisagi/Documents/Journals/Journal_of_Information_Systems/standardization_cache.json


In [3]:
# ==============================
# Cell 2 — Load Input + Validate
# ==============================

if not INPUT_CSV.exists():
    raise FileNotFoundError(f"Missing input CSV: {INPUT_CSV}")

data = pd.read_csv(INPUT_CSV)
total_rows = len(data)
print(f"Loaded rows: {total_rows}")

# Must-have base columns
base_required = ["Author_name", "Author_Address"]
missing = [c for c in base_required if c not in data.columns]
if missing:
    raise ValueError(f"Input missing columns: {missing}. Found: {list(data.columns)}")

# Optional standardized columns (from basic cleaning)
has_std_name = "Standardized_name" in data.columns
has_std_addr = "Standardized_Address" in data.columns
print("Has Standardized_name   :", has_std_name)
print("Has Standardized_Address:", has_std_addr)

Loaded rows: 4718
Has Standardized_name   : True
Has Standardized_Address: True


In [4]:
# =====================================
# Cell 3 — Helper Functions (Key + Text)
# =====================================

def _cell(x) -> str:
    if x is None:
        return ""
    if isinstance(x, float) and pd.isna(x):
        return ""
    s = str(x).strip()
    return "" if s.lower() == "nan" else s

def norm_whitespace(s: str) -> str:
    s = _cell(s).lower()
    s = re.sub(r"\s+", " ", s).strip()
    return s

def norm_address(addr: str) -> str:
    s = norm_whitespace(addr)
    s = re.sub(r"\s*,\s*", ", ", s)
    s = re.sub(r"\s*;\s*", "; ", s)
    s = re.sub(r"\s*\.\s*", ". ", s).strip()
    s = re.sub(r"(, ){2,}", ", ", s)
    return s

def norm_author(author: str) -> str:
    s = norm_whitespace(author)
    s = re.sub(r"[^a-z\s\-']", "", s).strip()
    s = re.sub(r"\s+", " ", s)
    return s

def get_author_for_api(row) -> str:
    # Prefer standardized name from basic cleaning, else original
    if "Standardized_name" in row and _cell(row["Standardized_name"]):
        return _cell(row["Standardized_name"])
    return _cell(row.get("Author_name", ""))

def get_address_for_api(row) -> str:
    # Prefer standardized address from basic cleaning, else original
    if "Standardized_Address" in row and _cell(row["Standardized_Address"]):
        return _cell(row["Standardized_Address"])
    return _cell(row.get("Author_Address", ""))

def cache_key(author: str, addr: str) -> str:
    return f"{norm_author(author)}||{norm_address(addr)}"

def build_item(author: str, addr: str) -> dict:
    # Structured item gives more reliable extraction than a concatenated string
    return {"author": author, "address": addr}

print("Helpers ready.")

Helpers ready.


In [5]:
# ======================================
# Cell 4 — Prompt + Batch Extractor (API)
# ======================================

SYSTEM_PROMPT = """
You extract and standardize author/university/location from input.
Return ONLY valid JSON.

You will receive JSON with key "items" (an array).
For each item, return one object in "results" in the SAME ORDER.

Each result MUST include these keys:
- Standardized_Author
- University
- Department
- State
- Country
- Pincode

Rules:
- English only
- Standardized_Author in Title Case (e.g., "John A Smith")
- University must be full name (no abbreviations)
- Country must be full name (e.g., "United States")
- If missing, use "" (empty string)
""".strip()

REQUIRED_KEYS = ["Standardized_Author","University","Department","State","Country","Pincode"]

def _sanitize(obj: dict) -> dict:
    obj = obj or {}
    out = {}
    for k in REQUIRED_KEYS:
        v = obj.get(k, "")
        if v is None:
            v = ""
        v = str(v).strip()
        out[k] = v
    return out

def extract_batch(items: list[dict], max_retries: int = 2) -> list[dict]:
    if not items:
        return []

    payload = {"items": items}
    attempt = 0
    backoff = 10
    last_err = None

    while attempt <= max_retries:
        try:
            resp = client.chat.completions.create(
                model=MODEL,
                messages=[
                    {"role":"system","content":SYSTEM_PROMPT},
                    {"role":"user","content":json.dumps(payload, ensure_ascii=False)}
                ],
                temperature=0,
                response_format={"type":"json_object"},
            )
            raw = (resp.choices[0].message.content or "").strip()
            parsed = json.loads(raw)
            results = parsed.get("results", [])

            out_list = []
            for i in range(len(items)):
                r = results[i] if i < len(results) else {}
                out_list.append(_sanitize(r))
            return out_list

        except Exception as e:
            last_err = e
            attempt += 1
            if attempt > max_retries:
                break
            time.sleep(backoff)
            backoff = min(backoff * 2, 120)

    print("Batch failed after retries:", repr(last_err))
    fallback = []
    for it in items:
        fallback.append({
            "Standardized_Author": _cell(it.get("author", "")),
            "University": "",
            "Department": "",
            "State": "",
            "Country": "",
            "Pincode": ""
        })
    return fallback

print("API extractor ready.")

API extractor ready.


In [6]:
# ==========================
# Cell 5 — Load / Save Cache
# ==========================

def load_cache(path: Path) -> dict:
    if not path.exists():
        return {}
    with open(path, "r", encoding="utf-8") as f:
        obj = json.load(f)
    cache = {}
    for k, v in obj.items():
        cache[str(k)] = _sanitize(v)
    return cache

def save_cache(path: Path, cache: dict) -> None:
    tmp = path.with_suffix(".tmp")
    with open(tmp, "w", encoding="utf-8") as f:
        json.dump(cache, f, indent=0, ensure_ascii=False)
    tmp.replace(path)

cache = load_cache(CACHE_JSON)
print(f"Cache loaded: {len(cache)} keys.")

Cache loaded: 210 keys.


In [7]:
# ===================================================
# Cell 6 — Build Unique Keys (Deduplicate API Calls)
# ===================================================

item_by_key = {}
rows_by_key = {}

for i, row in data.iterrows():
    author = get_author_for_api(row)
    addr   = get_address_for_api(row)
    k = cache_key(author, addr)

    if k not in rows_by_key:
        rows_by_key[k] = []
        item_by_key[k] = build_item(author, addr)

    rows_by_key[k].append(i)

unique_keys = list(rows_by_key.keys())

def non_empty(it: dict) -> bool:
    return bool(_cell(it.get("author")) or _cell(it.get("address")))

keys_needing_api = [k for k in unique_keys if non_empty(item_by_key[k])]
pending = [k for k in keys_needing_api if k not in cache]

MAX_PAIRS_PER_RUN = MAX_CALLS_PER_RUN * BATCH_SIZE
to_fetch = pending[:MAX_PAIRS_PER_RUN]
num_calls = (len(to_fetch) + BATCH_SIZE - 1) // BATCH_SIZE

print(f"Total rows: {total_rows}")
print(f"Unique keys: {len(unique_keys)}")
print(f"Non-empty keys: {len(keys_needing_api)}")
print(f"Pending keys: {len(pending)}")
print(f"This run: {len(to_fetch)} keys in {num_calls} API call(s) (max {MAX_CALLS_PER_RUN}).")

Total rows: 4718
Unique keys: 3838
Non-empty keys: 3837
Pending keys: 3627
This run: 200 keys in 20 API call(s) (max 20).


In [11]:
# ==========================================
# Cell 7 — Call API in Batches (Save per batch)
# ==========================================

processed = 0

for start in range(0, len(to_fetch), BATCH_SIZE):
    batch_keys  = to_fetch[start:start + BATCH_SIZE]
    batch_items = [item_by_key[k] for k in batch_keys]

    results = extract_batch(batch_items, max_retries=2)

    for k, res in zip(batch_keys, results):
        cache[k] = _sanitize(res)

    processed += len(batch_keys)
    save_cache(CACHE_JSON, cache)

    print(f"Cached {processed}/{len(to_fetch)} keys (saved).")

    time.sleep(DELAY_BETWEEN_CALLS)

print("Fetch complete.")
print("Cache size now:", len(cache))

Cached 10/200 keys (saved).
Cached 20/200 keys (saved).
Cached 30/200 keys (saved).
Cached 40/200 keys (saved).
Cached 50/200 keys (saved).
Cached 60/200 keys (saved).
Cached 70/200 keys (saved).
Cached 80/200 keys (saved).
Cached 90/200 keys (saved).
Cached 100/200 keys (saved).
Cached 110/200 keys (saved).
Cached 120/200 keys (saved).
Cached 130/200 keys (saved).
Cached 140/200 keys (saved).
Cached 150/200 keys (saved).
Cached 160/200 keys (saved).
Cached 170/200 keys (saved).
Cached 180/200 keys (saved).
Cached 190/200 keys (saved).
Cached 200/200 keys (saved).
Fetch complete.
Cache size now: 210


In [8]:
print("Processed so far:", len(cache))
print("Remaining:", len(unique_keys) - len(cache))

Processed so far: 210
Remaining: 3628


In [ ]:
for k in unique_keys:
    if k not in cache:
        it = item_by_key[k]
        cache[k] = {
            "Standardized_Author": _cell(it.get("author", "")),
            "University": "",
            "Department": "",
            "State": "",
            "Country": "",
            "Pincode": ""
        }

save_cache(CACHE_JSON, cache)
print("Cache finalized.")

In [ ]:
base_cols = ["URL","Journal_Title","Article_Title","Volume_Issue","Month_Year","Abstract","Keywords","Author_name","Author_email","Author_Address"]
for c in base_cols:
    if c not in data.columns:
        data[c] = None

HEADER = [
    "URL","Journal_Title","Article_Title","Volume_Issue","Month_Year","Abstract","Keywords",
    "Author_name","Standardized_Author","Author_email","Author_Address",
    "Standardized_University","Author_Department","Author_State","Author_Country","Author_Pincode",
    "Basic_Standardized_name","Basic_Standardized_Address"
]

with open(OUTPUT_CSV, "w", newline="", encoding="utf-8") as f:
    w = csv.writer(f)
    w.writerow(HEADER)

    for i in range(total_rows):
        row = data.iloc[i]

        author_api = get_author_for_api(row)
        addr_api   = get_address_for_api(row)
        k = cache_key(author_api, addr_api)

        res = cache.get(k, {})
        res = _sanitize(res)

        basic_std_name = _cell(row.get("Standardized_name", ""))
        basic_std_addr = _cell(row.get("Standardized_Address", ""))

        w.writerow([
            _cell(row.get("URL")),
            _cell(row.get("Journal_Title")),
            _cell(row.get("Article_Title")),
            _cell(row.get("Volume_Issue")),
            _cell(row.get("Month_Year")),
            _cell(row.get("Abstract")),
            _cell(row.get("Keywords")),
            _cell(row.get("Author_name")),
            _cell(res.get("Standardized_Author","")),
            _cell(row.get("Author_email")),
            _cell(row.get("Author_Address")),
            _cell(res.get("University","")),
            _cell(res.get("Department","")),
            _cell(res.get("State","")),
            _cell(res.get("Country","")),
            _cell(res.get("Pincode","")),
            basic_std_name,
            basic_std_addr
        ])

print(f"Wrote {total_rows} rows to {OUTPUT_CSV}.")

In [ ]:
out_df = pd.read_csv(OUTPUT_CSV)
print("Output shape:", out_df.shape)

check_cols = ["Standardized_Author","Standardized_University","Author_Country"]
print("Missing counts:")
print(out_df[check_cols].isnull().sum())

out_df.head(10)